In [1]:
from datasets import load_dataset

dataset = load_dataset("coastalcph/lex_glue", "unfair_tos")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 5532
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 1607
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 2275
    })
})


In [2]:
#Example
example = dataset["train"][0]
print("Text:", example["text"])
print("Labels:", example["labels"])

Text: notice to california subscribers : you may cancel your subscription , without penalty or obligation , at any time prior to midnight of the third business day following the date you subscribed . 

Labels: []


In [3]:
# What the label numbers correspond to
label_names = dataset["train"].features["labels"].feature.names
print(label_names)

['Limitation of liability', 'Unilateral termination', 'Unilateral change', 'Content removal', 'Contract by using', 'Choice of law', 'Jurisdiction', 'Arbitration']


### Class Distribution

In [4]:
import pandas as pd
from collections import Counter

train_df = dataset["train"].to_pandas()
print(train_df.shape)
train_df.head()

(5532, 2)


,text,labels
0,notice to california subscribers : you may can...,[]
1,"if you subscribed using your apple id , refund...",[]
2,"if you wish to request a refund , please visit...",[]
3,if you subscribed using your google play store...,[]
4,key changes in this version : we 've included ...,[]


In [5]:
label_names = dataset["train"].features["labels"].feature.names

# Count how many times each category appears across all training sentences
flat_labels = [label for labels in train_df["labels"] for label in labels]
label_counts = Counter(flat_labels)

for idx, name in enumerate(label_names):
    print(f"{name}: {label_counts.get(idx, 0)}")

# How many sentences have NO unfair label at all?
no_label_count = sum(1 for labels in train_df["labels"] if len(labels) == 0)
print(f"\nSentences with NO unfair label: {no_label_count} ({no_label_count/len(train_df)*100:.1f}%)")

Limitation of liability: 191
Unilateral termination: 139
Unilateral change: 122
Content removal: 73
Contract by using: 76
Choice of law: 39
Jurisdiction: 34
Arbitration: 28

Sentences with NO unfair label: 4902 (88.6%)


### Sentence Length Distribution

In [6]:
train_df["word_count"] = train_df["text"].apply(lambda x: len(x.split()))

print(train_df["word_count"].describe())

count    5532.000000
mean       32.283080
std        24.745549
min         6.000000
25%        17.000000
50%        26.000000
75%        39.000000
max       441.000000
Name: word_count, dtype: float64


In [7]:
# Bucket sentences into length ranges so we can visualize the distribution
bins = [0, 10, 20, 30, 50, 75, 100, 1000]
bin_labels = ["1-10", "11-20", "21-30", "31-50", "51-75", "76-100", "100+"]

train_df["length_bucket"] = pd.cut(train_df["word_count"], bins=bins, labels=bin_labels)
bucket_counts = train_df["length_bucket"].value_counts().sort_index()

print(bucket_counts)

length_bucket
1-10       497
11-20     1375
21-30     1402
31-50     1447
51-75      543
76-100     151
100+       117
Name: count, dtype: int64


### Label conversion to required format

In [8]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

# Get the validation and test splits as DataFrames too, same way we did for train
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

# Convert label lists into multi-hot binary vectors
mlb = MultiLabelBinarizer(classes=range(len(label_names)))
y_train = mlb.fit_transform(train_df["labels"])
y_val = mlb.transform(val_df["labels"])
y_test = mlb.transform(test_df["labels"])

print(y_train.shape)
print(y_train[:5])

(5532, 8)
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]


### TF-IDF Vectoriztion

In [9]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train = vectorizer.fit_transform(train_df["text"])
X_val = vectorizer.transform(val_df["text"])
X_test = vectorizer.transform(test_df["text"])

print(X_train.shape)

important_terms = ["arbitration", "jurisdiction", "liability", "terminate", "unilateral"]
for term in important_terms:
    status = "in vocabulary" if term in vectorizer.vocabulary_ else "MISSING"
    print(f"{term} → {status}")

(5532, 10000)
arbitration → in vocabulary
jurisdiction → in vocabulary
liability → in vocabulary
terminate → in vocabulary
unilateral → MISSING


In [10]:
matches = train_df[train_df["text"].str.contains("unilateral", case=False)]
print(f"Sentences containing 'unilateral' (any form): {len(matches)}")
print(matches["text"].head(5).tolist())

Sentences containing 'unilateral' (any form): 0
[]


### Model

In [11]:
model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

                         precision    recall  f1-score   support

Limitation of liability       0.53      0.87      0.66        38
 Unilateral termination       0.52      0.84      0.65        38
      Unilateral change       0.52      0.68      0.59        38
        Content removal       0.31      0.92      0.46        13
      Contract by using       0.45      0.78      0.57        23
          Choice of law       0.71      0.92      0.80        13
           Jurisdiction       0.86      0.75      0.80        16
            Arbitration       0.33      0.86      0.48         7

              micro avg       0.50      0.81      0.62       186
              macro avg       0.53      0.83      0.63       186
           weighted avg       0.53      0.81      0.63       186
            samples avg       0.08      0.09      0.08       186



In [12]:
model_unweighted = OneVsRestClassifier(
    LogisticRegression(max_iter=1000)
)
model_unweighted.fit(X_train, y_train)
y_pred_unweighted = model_unweighted.predict(X_test)

print("=== WITHOUT class_weight='balanced' ===")
print(classification_report(y_test, y_pred_unweighted, target_names=label_names, zero_division=0))

=== WITHOUT class_weight='balanced' ===
                         precision    recall  f1-score   support

Limitation of liability       0.83      0.26      0.40        38
 Unilateral termination       1.00      0.18      0.31        38
      Unilateral change       1.00      0.16      0.27        38
        Content removal       1.00      0.15      0.27        13
      Contract by using       1.00      0.04      0.08        23
          Choice of law       0.00      0.00      0.00        13
           Jurisdiction       0.00      0.00      0.00        16
            Arbitration       0.00      0.00      0.00         7

              micro avg       0.93      0.14      0.24       186
              macro avg       0.60      0.10      0.17       186
           weighted avg       0.77      0.14      0.23       186
            samples avg       0.02      0.02      0.02       186



### Saving the baseline model

In [13]:
import joblib

joblib.dump(model, "../models/baseline_logreg.pkl")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")
joblib.dump(mlb, "../models/label_binarizer.pkl")

print("Saved baseline model, vectorizer, and label binarizer to models/")

Saved baseline model, vectorizer, and label binarizer to models/


In [14]:
report_text = classification_report(y_test, y_pred, target_names=label_names, zero_division=0)

with open("../models/baseline_metrics.txt", "w") as f:
    f.write("Classical TF-IDF + Logistic Regression (class_weight='balanced') baseline\n\n")
    f.write(report_text)

print("Saved baseline metrics to models/baseline_metrics.txt")

Saved baseline metrics to models/baseline_metrics.txt


### Transformers

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample_text = train_df["text"].iloc[0]
tokens = tokenizer(sample_text)
print(tokens)

{'input_ids': [101, 5060, 2000, 2662, 17073, 1024, 2017, 2089, 17542, 2115, 15002, 1010, 2302, 6531, 2030, 14987, 1010, 2012, 2151, 2051, 3188, 2000, 7090, 1997, 1996, 2353, 2449, 2154, 2206, 1996, 3058, 2017, 4942, 29234, 2094, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


### Tokenizing the whole dataset

In [16]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_encodings = tokenizer(
    list(train_df["text"]),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

val_encodings = tokenizer(
    list(val_df["text"]),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

test_encodings = tokenizer(
    list(test_df["text"]),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(train_encodings["input_ids"].shape)

torch.Size([5532, 128])


In [17]:
import torch

class ToSDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

train_dataset = ToSDataset(train_encodings, y_train)
val_dataset = ToSDataset(val_encodings, y_val)
test_dataset = ToSDataset(test_encodings, y_test)

print(train_dataset[0])

{'input_ids': tensor([  101,  5060,  2000,  2662, 17073,  1024,  2017,  2089, 17542,  2115,
        15002,  1010,  2302,  6531,  2030, 14987,  1010,  2012,  2151,  2051,
         3188,  2000,  7090,  1997,  1996,  2353,  2449,  2154,  2206,  1996,
         3058,  2017,  4942, 29234,  2094,  1012,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 

In [18]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score

model_transformer = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_names),
    problem_type="multi_label_classification"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = (logits > 0).astype(int)
    macro_f1 = f1_score(labels, predictions, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, predictions, average="micro", zero_division=0)
    return {"macro_f1": macro_f1, "micro_f1": micro_f1}

training_args = TrainingArguments(
    output_dir="../models/transformer_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
)

trainer = Trainer(
    model=model_transformer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer set up. Ready to train.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainer set up. Ready to train.


In [19]:
trainer.train()

C:\Users\AKSHAT\Python_projects\contract-risk-assistant\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [20]:
import os
print(os.listdir("../models/transformer_checkpoints"))

['checkpoint-1384', 'checkpoint-2076', 'checkpoint-692']


In [22]:
trainer = Trainer(
    model=model_transformer,
    args=training_args,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

output = trainer.predict(test_dataset)
logits = output.predictions
y_pred_transformer = (logits > 0).astype(int)

report_text_transformer = classification_report(
    y_test, y_pred_transformer, target_names=label_names, zero_division=0
)
print(report_text_transformer)

                         precision    recall  f1-score   support

Limitation of liability       0.68      0.71      0.69        38
 Unilateral termination       0.77      0.71      0.74        38
      Unilateral change       0.75      0.71      0.73        38
        Content removal       0.77      0.77      0.77        13
      Contract by using       0.83      0.65      0.73        23
          Choice of law       1.00      0.92      0.96        13
           Jurisdiction       1.00      0.56      0.72        16
            Arbitration       0.83      0.71      0.77         7

              micro avg       0.78      0.71      0.74       186
              macro avg       0.83      0.72      0.76       186
           weighted avg       0.79      0.71      0.74       186
            samples avg       0.08      0.08      0.08       186



In [25]:
### Saving the model
import joblib

trainer.save_model("../models/transformer_final")
tokenizer.save_pretrained("../models/transformer_final")

with open("../models/transformer_metrics.txt", "w") as f:
    f.write("Fine-tuned DistilBERT (3 epochs) test set results\n\n")
    f.write(report_text_transformer)

print("Saved final transformer model, tokenizer, and metrics.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final transformer model, tokenizer, and metrics.


### Interpretibility with SHAP

In [1]:
#reloading the models
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_transformer = AutoModelForSequenceClassification.from_pretrained("../models/transformer_final")
tokenizer = AutoTokenizer.from_pretrained("../models/transformer_final")

print("Model and tokenizer loaded from disk.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model and tokenizer loaded from disk.


In [4]:
label_names = ['Limitation of liability', 'Unilateral termination', 'Unilateral change',
               'Content removal', 'Contract by using', 'Choice of law', 'Jurisdiction', 'Arbitration']

In [6]:
import numpy as np
import torch

def predict_proba(texts):
    inputs = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs = model_transformer(**inputs)
    probabilities = torch.sigmoid(outputs.logits).numpy()
    return probabilities

# Quick manual test before wiring up SHAP
test_sentence = ["we may terminate your account at any time for any reason without notice"]
probs = predict_proba(test_sentence)
for name, prob in zip(label_names, probs[0]):
    print(f"{name}: {prob:.3f}")

Limitation of liability: 0.083
Unilateral termination: 0.924
Unilateral change: 0.062
Content removal: 0.080
Contract by using: 0.013
Choice of law: 0.008
Jurisdiction: 0.007
Arbitration: 0.011


### SHAP Explainer

In [7]:
def explain_prediction(sentence, category_name):
    words = sentence.split()
    label_idx = label_names.index(category_name)

    baseline_prob = predict_proba([sentence])[0][label_idx]

    importances = []
    for i in range(len(words)):
        modified_words = words[:i] + words[i+1:]
        modified_sentence = " ".join(modified_words)
        modified_prob = predict_proba([modified_sentence])[0][label_idx]
        importance = baseline_prob - modified_prob
        importances.append((words[i], importance))

    print(f"Baseline probability for '{category_name}': {baseline_prob:.3f}\n")
    for word, imp in sorted(importances, key=lambda x: -abs(x[1])):
        direction = "pushes UP" if imp > 0 else "pushes DOWN"
        print(f"  {word:20s} {direction} by {abs(imp):.3f}")

explain_prediction(
    "we may terminate your account at any time for any reason without notice",
    "Unilateral termination"
)

Baseline probability for 'Unilateral termination': 0.924

  terminate            pushes UP by 0.741
  notice               pushes UP by 0.013
  without              pushes UP by 0.007
  may                  pushes UP by 0.005
  your                 pushes DOWN by 0.003
  account              pushes UP by 0.003
  time                 pushes UP by 0.002
  for                  pushes UP by 0.002
  reason               pushes UP by 0.002
  any                  pushes UP by 0.001
  any                  pushes UP by 0.001
  at                   pushes UP by 0.001
  we                   pushes UP by 0.001


In [10]:
#Reloading the dataset
from datasets import load_dataset

dataset = load_dataset("coastalcph/lex_glue", "unfair_tos")
test_df = dataset["test"].to_pandas()

print(test_df.shape)

(1607, 2)


In [11]:
# Grab a real example the model correctly flagged, for a category we care about
real_example_idx = test_df[test_df["labels"].apply(lambda l: label_names.index("Choice of law") in l)].index[0]
real_sentence = test_df.loc[real_example_idx, "text"]

print("Real test sentence:", real_sentence)
print()
explain_prediction(real_sentence, "Choice of law")

Real test sentence: these terms and any action related thereto will be governed by the laws of the state of california without regard to its conflict of laws provisions . 


Baseline probability for 'Choice of law': 0.798

  california           pushes UP by 0.057
  conflict             pushes UP by 0.025
  by                   pushes UP by 0.020
  governed             pushes UP by 0.011
  of                   pushes UP by 0.008
  any                  pushes UP by 0.007
  regard               pushes UP by 0.006
  the                  pushes UP by 0.006
  provisions           pushes DOWN by 0.005
  thereto              pushes UP by 0.004
  to                   pushes UP by 0.004
  and                  pushes UP by 0.003
  .                    pushes UP by 0.003
  of                   pushes UP by 0.003
  without              pushes DOWN by 0.003
  terms                pushes UP by 0.002
  laws                 pushes UP by 0.002
  be                   pushes UP by 0.002
  its            

In [12]:
#Saving the analysis
with open("../models/interpretability_examples.txt", "w", encoding="utf-8") as f:
    f.write("Interpretability Analysis: Leave-One-Out Word Importance\n")
    f.write("=" * 60 + "\n\n")
    f.write("Example 1: Synthetic sentence, category 'Unilateral termination'\n")
    f.write("Sentence: we may terminate your account at any time for any reason without notice\n")
    f.write("Finding: single dominant word ('terminate', contributing 0.741 of 0.924 probability)\n\n")
    f.write("Example 2: Real test-set sentence, category 'Choice of law'\n")
    f.write(f"Sentence: {real_sentence}\n")
    f.write("Finding: importance distributed across several jurisdiction-related words\n")
    f.write("('california', 'conflict', 'governed', 'regard') rather than one dominant word\n")

print("Saved interpretability findings.")

Saved interpretability findings.
